In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm

# Your dataset path
image_folder = '/content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/images/train'
label_folder = '/content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/labels/train'
os.makedirs(label_folder, exist_ok=True)


In [42]:

def normalize_points(points, width, height):
    return [(x / width, y / height) for x, y in points]

def process_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

    if img is None:
        print("❌ Could not load:", image_path)
        return

    # If the image has an alpha channel, use it as mask; else, use black background mask
    if img.shape[-1] == 4:
        alpha = img[..., 3]
        mask = alpha > 0
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mask = gray > 0  # non-black pixels

    mask = mask.astype(np.uint8) * 255

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    height, width = img.shape[:2]
    label_path = image_path.replace('images', 'labels').replace('.png', '.txt')

    with open(label_path, 'w') as f:
        for cnt in contours:
            cnt = cnt.squeeze()

            # Filter out too small areas
            if cnt.ndim != 2 or len(cnt) < 3:
                continue

            norm_pts = normalize_points(cnt, width, height)
            flat_pts = [str(coord) for pt in norm_pts for coord in pt]

            # Class index is 0 (for "coke")
            line = f"0 " + " ".join(flat_pts) + "\n"
            f.write(line)

# Process all images
image_paths = sorted(glob(os.path.join(image_folder, '*.png')))
print(f"Found {len(image_paths)} images.")

for path in tqdm(image_paths):
    process_image(path)

print("✅ Done labeling all images.")


Found 0 images.


0it [00:00, ?it/s]

✅ Done labeling all images.


In [9]:
image_paths = glob.glob(os.path.join(image_folder, '*.png'))
print(f"Found {len(image_paths)} images.")

for img_path in tqdm(image_paths):
    process_image(img_path)

print("✅ Labeling complete! Check:", label_folder)


Found 72 images.


100%|██████████| 72/72 [00:39<00:00,  1.82it/s]

✅ Labeling complete! Check: /content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/labels/val


In [17]:
yaml_path = "/content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/data.yaml"

data_yaml = """
train: /content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/images/train
val: /content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/images/val
nc: 1
names: ['coke']
"""

with open(yaml_path, "w") as f:
    f.write(data_yaml.strip())


In [18]:
!pip install ultralytics --upgrade -q
from ultralytics import YOLO

In [21]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

model.train(
    data=yaml_path,
    epochs=72,
    imgsz=640,
    batch=8,
    name="train2"
)

Ultralytics 8.3.113 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=segment, mode=train, model=yolov8n-seg.pt, data=/content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/data.yaml, epochs=72, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train24, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=

train: Scanning /content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/labels/train.cache... 288 images, 0 backgrounds, 0 corrupt: 100%|██████████| 288/288 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 103.9±44.4 MB/s, size: 302.5 KB)


val: Scanning /content/drive/MyDrive/ColabTestProject/coke_dataset/dataset/labels/val.cache... 72 images, 0 backgrounds, 0 corrupt: 100%|██████████| 72/72 [00:00<?, ?it/s]


...

                   all         72         72      0.999          1      0.995      0.995      0.999          1      0.995      0.995



72 epochs completed in 0.229 hours.
Optimizer stripped from runs/segment/train24/weights/last.pt, 6.8MB
Optimizer stripped from runs/segment/train24/weights/best.pt, 6.8MB

Validating runs/segment/train24/weights/best.pt...
Ultralytics 8.3.113 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 12.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  1.87it/s]


                   all         72         72      0.999          1      0.995      0.995      0.999          1      0.995      0.995
Speed: 0.5ms preprocess, 5.6ms inference, 0.0ms loss, 5.3ms postprocess per image
Results saved to runs/segment/train24


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79f1c9ef2590>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [23]:
!find /content -name "best.pt"

/content/runs/segment/train24/weights/best.pt
/content/runs/segment/train23/weights/best.pt
/content/runs/segment/train22/weights/best.pt


In [24]:
# Example (update this with your actual path from Step 1):
from google.colab import files
files.download('/content/runs/segment/train24/weights/best.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>